# 🏏 IPL Advanced SQL Challenge — Databricks Notebook

**Difficulty:** Hard  
**Tables used:** `Match`, `Player_match`  
**Concepts:** CTEs, Window Functions (`RANK`), Anti-join, Conditional aggregation

---

## ❓ Question

> Find players who were **Man of the Match in at least 3 different venues**, but have **NEVER won a match as captain**.  
> For each such player show:
> - Total MOM awards
> - The season they peaked (most MOM awards in a single season)
> - Their win rate % (matches won / total matches played)
>
> Break ties in peak season by picking the **most recent season**. Exclude no-result matches.

---
## 🔧 Cell 1 — Load all tables from Volume

In [ ]:
ball_by_ball_df = spark.read.csv("/Volumes/workspace/karan/ipl/Ball_By_Ball.csv/", header=True, inferSchema=True)
match_df        = spark.read.csv("/Volumes/workspace/karan/ipl/Match.csv",           header=True, inferSchema=True)
player_df       = spark.read.csv("/Volumes/workspace/karan/ipl/Player.csv/",          header=True, inferSchema=True)
team_df         = spark.read.csv("/Volumes/workspace/karan/ipl/Team.csv",             header=True, inferSchema=True)
player_match_df = spark.read.csv("/Volumes/workspace/karan/ipl/Player_match.csv",     header=True, inferSchema=True)

ball_by_ball_df.createOrReplaceTempView("Ball_By_Ball")
match_df.createOrReplaceTempView("Match")
player_df.createOrReplaceTempView("Player")
team_df.createOrReplaceTempView("Team")
player_match_df.createOrReplaceTempView("Player_match")

print("✅ All tables registered as temp views")

---
## 🔍 Cell 2 — Quick schema check (optional)

In [ ]:
print("=== Match columns ===")
print(match_df.columns)

print("\n=== Player_match columns ===")
print(player_match_df.columns)

print(f"\nMatch rows       : {match_df.count()}")
print(f"Player_match rows: {player_match_df.count()}")

---
## ✅ Cell 3 — Solution Query

**CTE breakdown:**
| CTE | Purpose |
|-----|---------|
| `clean_players` | Remove dummy row (`Player_Id = -1`) and no-result matches |
| `captains_who_won` | Build exclusion blacklist — players who won AT LEAST ONCE as captain |
| `mom_stats` | Count MOM awards + distinct venues; filter to 3+ venues via `HAVING` |
| `peak_season` | `RANK()` within each player by MOM count, tie-break = most recent season |
| `win_rate` | Win % across ALL their matches (not just MOM ones) |

In [ ]:
result = spark.sql("""

WITH

-- Step 1: Clean baseline — remove dummy rows & no-result matches
clean_players AS (
  SELECT p.*
  FROM   Player_match p
  JOIN   Match m  ON p.Match_Id = m.match_id
  WHERE  p.Player_Id    != -1
  AND    m.Outcome_Type  = 'Result'
),

-- Step 2: Blacklist — players who won AT LEAST ONCE as captain
-- Gotcha: Player_Captain stores the captain's NAME, not Player_Id
captains_who_won AS (
  SELECT DISTINCT Player_Id
  FROM   clean_players
  WHERE  Player_Captain     = Player_Name  -- they were their team's captain
  AND    IsPlayers_Team_won = 1            -- and their team won
),

-- Step 3: MOM stats — count awards AND distinct venues per player
mom_stats AS (
  SELECT
    p.Player_Id,
    p.Player_Name,
    COUNT(*)                        AS total_mom_awards,
    COUNT(DISTINCT m.Venue_Name)    AS distinct_venues
  FROM   clean_players p
  JOIN   Match m  ON p.Match_Id = m.match_id
  WHERE  p.is_manofThematch = 1
  GROUP BY p.Player_Id, p.Player_Name
  HAVING COUNT(DISTINCT m.Venue_Name) >= 3   -- core filter: MOM at 3+ different venues
),

-- Step 4: MOM count by season — to find each player's peak season
-- RANK() assigns same rank to ties; tie-break = most recent season (Season_year DESC)
peak_season AS (
  SELECT Player_Id, Season_year AS peak_season, mom_count AS peak_mom_count
  FROM (
    SELECT
      Player_Id,
      Season_year,
      COUNT(*) AS mom_count,
      RANK() OVER (
        PARTITION BY Player_Id
        ORDER BY COUNT(*) DESC, Season_year DESC  -- tiebreak: latest season wins
      ) AS rnk
    FROM   clean_players
    WHERE  is_manofThematch = 1
    GROUP BY Player_Id, Season_year
  )
  WHERE rnk = 1
),

-- Step 5: Win rate across ALL matches (not just MOM ones)
-- Gotcha: IsPlayers_Team_won is read as STRING from CSV — CAST to INT before SUM
win_rate AS (
  SELECT
    Player_Id,
    COUNT(*)                                            AS total_matches,
    SUM(CAST(IsPlayers_Team_won AS INT))                AS matches_won,
    ROUND(
      SUM(CAST(IsPlayers_Team_won AS INT)) * 100.0
      / COUNT(*), 1
    )                                                   AS win_pct
  FROM  clean_players
  GROUP BY Player_Id
)

-- Final: Assemble all CTEs, exclude captains who ever won
SELECT
  ms.Player_Name,
  ms.total_mom_awards,
  ms.distinct_venues,
  ps.peak_season,
  ps.peak_mom_count,
  wr.total_matches,
  wr.matches_won,
  wr.win_pct                             AS win_rate_pct
FROM       mom_stats    ms
JOIN       peak_season  ps  ON ms.Player_Id = ps.Player_Id
JOIN       win_rate     wr  ON ms.Player_Id = wr.Player_Id
WHERE      ms.Player_Id NOT IN (SELECT Player_Id FROM captains_who_won)
ORDER BY   ms.total_mom_awards DESC, ms.distinct_venues DESC

""")

result.show(truncate=False)

---
## 📊 Cell 4 — Display as formatted table (nicer output)

In [ ]:
display(result)

---
## ⚠️ Cell 5 — Gotchas explained

| # | Gotcha | Why it matters |
|---|--------|----------------|
| 1 | `Player_Captain` stores a **name**, not an ID | Joining on `Player_Captain = Player_Name` is the only way to identify captains |
| 2 | Dummy row `Player_Id = -1` | Pollutes aggregations — must be filtered in the first CTE |
| 3 | `Outcome_Type = 'Result'` filter | Ties and No-Result matches skew win rates — exclude them early |
| 4 | `RANK()` vs `ROW_NUMBER()` | `RANK()` assigns same rank to tied seasons; combined with `Season_year DESC` tiebreak, the latest season correctly surfaces |
| 5 | `CAST(IsPlayers_Team_won AS INT)` | CSV reads all columns as strings; `SUM()` on a string returns NULL silently |
| 6 | `NOT IN` with NULLs | If `captains_who_won` ever contains a NULL Player_Id, `NOT IN` returns zero rows. Safe alternative: use `NOT EXISTS` or `LEFT JOIN ... WHERE ... IS NULL` |

---
## 🛡️ Cell 6 — NULL-safe version (production-grade)

Replaces `NOT IN` with a `LEFT JOIN ... WHERE IS NULL` anti-join pattern — safe even if Player_Id contains NULLs.

In [ ]:
result_safe = spark.sql("""

WITH

clean_players AS (
  SELECT p.*
  FROM   Player_match p
  JOIN   Match m  ON p.Match_Id = m.match_id
  WHERE  p.Player_Id    != -1
  AND    m.Outcome_Type  = 'Result'
),

captains_who_won AS (
  SELECT DISTINCT Player_Id
  FROM   clean_players
  WHERE  Player_Captain     = Player_Name
  AND    IsPlayers_Team_won = 1
),

mom_stats AS (
  SELECT
    p.Player_Id,
    p.Player_Name,
    COUNT(*)                        AS total_mom_awards,
    COUNT(DISTINCT m.Venue_Name)    AS distinct_venues
  FROM   clean_players p
  JOIN   Match m  ON p.Match_Id = m.match_id
  WHERE  p.is_manofThematch = 1
  GROUP BY p.Player_Id, p.Player_Name
  HAVING COUNT(DISTINCT m.Venue_Name) >= 3
),

peak_season AS (
  SELECT Player_Id, Season_year AS peak_season, mom_count AS peak_mom_count
  FROM (
    SELECT
      Player_Id, Season_year,
      COUNT(*) AS mom_count,
      RANK() OVER (
        PARTITION BY Player_Id
        ORDER BY COUNT(*) DESC, Season_year DESC
      ) AS rnk
    FROM   clean_players
    WHERE  is_manofThematch = 1
    GROUP BY Player_Id, Season_year
  )
  WHERE rnk = 1
),

win_rate AS (
  SELECT
    Player_Id,
    COUNT(*)                                            AS total_matches,
    SUM(CAST(IsPlayers_Team_won AS INT))                AS matches_won,
    ROUND(
      SUM(CAST(IsPlayers_Team_won AS INT)) * 100.0
      / COUNT(*), 1
    )                                                   AS win_pct
  FROM  clean_players
  GROUP BY Player_Id
)

-- NULL-safe anti-join: LEFT JOIN + IS NULL instead of NOT IN
SELECT
  ms.Player_Name,
  ms.total_mom_awards,
  ms.distinct_venues,
  ps.peak_season,
  ps.peak_mom_count,
  wr.total_matches,
  wr.matches_won,
  wr.win_pct                             AS win_rate_pct
FROM       mom_stats    ms
JOIN       peak_season  ps   ON ms.Player_Id = ps.Player_Id
JOIN       win_rate     wr   ON ms.Player_Id = wr.Player_Id
LEFT JOIN  captains_who_won cww ON ms.Player_Id = cww.Player_Id
WHERE      cww.Player_Id IS NULL          -- exclude captains who won
ORDER BY   ms.total_mom_awards DESC, ms.distinct_venues DESC

""")

display(result_safe)

---
## 🧠 Mini Quiz

**Question:**  
In `Cell 3`, we used `NOT IN` for the exclusion. If even **one** `Player_Id` inside `captains_who_won` is `NULL`, the entire final result returns **zero rows**.  

- Why does this happen?
- How does `Cell 6` fix it?
- When would you still prefer `NOT IN` over `LEFT JOIN ... IS NULL`?

*(Hint: think about how SQL evaluates `x NOT IN (1, 2, NULL)` using three-valued logic)*